## Look up `<unknown:…>` kernels in the multicore C source

For each `*.by-source.txt` in this folder, parses the filename as `<spec>-<run_equi>-<system>-<backend>` and looks for the matching `<run_equi>.c` in `saved_c_files_for_local_profiling/`. Then, for every row whose provenance is `<unknown:NAME>`, it extracts the `static … NAME(…) { … }` definition from that `.c` and renders the body so you can read the inline source-location comments and infer what generated the kernel.

If a needed `.c` isn't in `saved_c_files_for_local_profiling/`, generate it with:
```
futhark multicore --library <run_equi>.fut
```
and move/copy the resulting `<run_equi>.c` into that folder.

In [ ]:
import re
from pathlib import Path
from IPython.display import display, Markdown

HERE = Path('.').resolve()
C_DIR = HERE / 'saved_c_files_for_local_profiling'

# Profile-event suffixes the multicore runtime appends (stripped when
# searching for the underlying C function).
_EVENT_SUFFIXES = ('_runs_total', '_runs', '_total')

# Window around each call site, in source lines.
CTX_BEFORE = 15
CTX_AFTER  = 10

def expected_c(by_source_path):
    """Parse filename as <spec>-<run_equi>-<system>-<backend>.by-source.txt and
    return C_DIR/<run_equi>.c. Returns None if no `run_…` segment is found."""
    stem = by_source_path.name[:-len('.by-source.txt')]
    for part in stem.split('-'):
        if part.startswith('run_'):
            return C_DIR / f'{part}.c'
    return None

def _extract_one(c_text, fn_name):
    """Return (matched_name, body) for `static <retty> [*]fn_name(...) {`
    (the definition, not a forward declaration); or (None, None)."""
    sig_pat = re.compile(
        r'^static\s+[\w\s\*]+?\b' + re.escape(fn_name) + r'\s*\(',
        re.MULTILINE,
    )
    for m in sig_pat.finditer(c_text):
        depth, i = 1, m.end()
        while i < len(c_text) and depth:
            if   c_text[i] == '(': depth += 1
            elif c_text[i] == ')': depth -= 1
            i += 1
        while i < len(c_text) and c_text[i].isspace():
            i += 1
        if i >= len(c_text) or c_text[i] != '{':
            continue
        depth, i = 1, i + 1
        while i < len(c_text) and depth:
            if   c_text[i] == '{': depth += 1
            elif c_text[i] == '}': depth -= 1
            i += 1
        return fn_name, c_text[m.start():i]
    return None, None

def _name_candidates(fn_name):
    """Profile-event names look like 'futhark_mc_<task>_<suffix>' where the C
    function is just '<task>'. Yield names to try, most specific first."""
    yield fn_name
    stripped = fn_name
    if stripped.startswith('futhark_mc_'):
        stripped = stripped[len('futhark_mc_'):]
        yield stripped
    for sfx in _EVENT_SUFFIXES:
        if stripped.endswith(sfx):
            yield stripped[:-len(sfx)]
            break

def extract_function(c_text, fn_name):
    """Return (matched_name, body) for the C function corresponding to the
    profile-event name fn_name. Tries exact match, then strips runtime
    prefix/suffixes, then falls back to wildcard match on the trailing
    numeric ID. Returns (None, None) if nothing is found.
    """
    seen = set()
    for cand in _name_candidates(fn_name):
        if cand in seen:
            continue
        seen.add(cand)
        name, body = _extract_one(c_text, cand)
        if body is not None:
            return name, body
    nums = re.findall(r'\d{3,}', fn_name)
    if nums:
        nid = nums[-1]
        wild = re.compile(
            r'^static\s+[\w\s\*]+?\b(\w*_' + re.escape(nid) + r')\s*\(',
            re.MULTILINE,
        )
        for m in wild.finditer(c_text):
            name, body = _extract_one(c_text, m.group(1))
            if body is not None:
                return name, body
    return None, None

def _enclosing_fn_name(c_text, offset):
    """Heuristic: name of the most recent `static … <name>(` defined before
    offset. Useful as a hint at which kernel/function is making the call."""
    pat = re.compile(r'^static\s+[\w\s\*]+?\b(\w+)\s*\(', re.MULTILINE)
    last = None
    for m in pat.finditer(c_text, 0, offset):
        last = m.group(1)
    return last

def find_call_sites(c_text, fn_name, defn_body=None):
    """Yield (offset, line_no, enclosing_fn) for each call site of fn_name in
    c_text, excluding the span of its own definition body (if provided).
    Deduplicates by line number."""
    pat = re.compile(r'\b' + re.escape(fn_name) + r'\s*\(')
    defn_start = defn_end = -1
    if defn_body:
        defn_start = c_text.find(defn_body)
        if defn_start >= 0:
            defn_end = defn_start + len(defn_body)
    seen_lines = set()
    for m in pat.finditer(c_text):
        if defn_start <= m.start() < defn_end:
            continue
        line_no = c_text.count('\n', 0, m.start()) + 1
        if line_no in seen_lines:
            continue
        seen_lines.add(line_no)
        yield m.start(), line_no, _enclosing_fn_name(c_text, m.start())

def line_window(c_text, offset, before=CTX_BEFORE, after=CTX_AFTER):
    """Return (start_line, joined_text) for an N-line window around offset."""
    lines = c_text.splitlines()
    idx = c_text.count('\n', 0, offset)
    lo = max(0, idx - before)
    hi = min(len(lines), idx + after + 1)
    return lo + 1, '\n'.join(lines[lo:hi])

def find_unknowns(by_source_path):
    """Return list of dicts for `<unknown:…>` rows in a by-source.txt."""
    rows = []
    with open(by_source_path) as f:
        for line in f:
            line = line.rstrip()
            if not line.startswith('<unknown:'):
                continue
            parts = line.rsplit(None, 4)
            if len(parts) != 5:
                continue
            src, nk, ct, ms, fr = parts
            kernel = src[len('<unknown:'):].rstrip('>')
            rows.append({
                'kernel':    kernel,
                '#kernels':  int(nk),
                'count':     int(ct),
                'sum_ms':    float(ms),
                'frac':      float(fr),
            })
    return rows

In [ ]:
files = sorted(HERE.glob('*.by-source.txt'))
if not files:
    print('No *.by-source.txt files in', HERE)

for p in files:
    display(Markdown(f'## `{p.name}`'))
    c_path = expected_c(p)
    if c_path is None:
        display(Markdown('_Could not infer matching `.c` from filename._'))
        continue
    if not c_path.exists():
        display(Markdown(
            f'_No `{c_path.name}` in `{C_DIR}` — run '
            f'`futhark multicore --library {c_path.stem}.fut` and place the '
            f'resulting `.c` into that folder._'
        ))
        continue
    c_text = c_path.read_text()
    unknowns = find_unknowns(p)
    if not unknowns:
        display(Markdown('_No `<unknown:…>` rows._'))
        continue
    unknowns.sort(key=lambda r: -r['sum_ms'])
    for u in unknowns:
        display(Markdown(
            f'### `{u["kernel"]}` — sum {u["sum_ms"]:.2f} ms, '
            f'frac {u["frac"]:.4f}, count {u["count"]}'
        ))
        matched, body = extract_function(c_text, u['kernel'])
        if body is None:
            display(Markdown(f'_Not found in `{c_path.name}`._'))
            continue
        if matched != u['kernel']:
            display(Markdown(f'_matched as_ `{matched}`'))
        display(Markdown(f'**Definition**'))
        display(Markdown(f'```c\n{body}\n```'))

        sites = list(find_call_sites(c_text, matched, defn_body=body))
        if not sites:
            display(Markdown('_(no other call sites found)_'))
            continue
        display(Markdown(f'**Call sites ({len(sites)})**'))
        for off, line_no, encl in sites:
            start_line, ctx = line_window(c_text, off)
            encl_note = f' (in `{encl}`)' if encl else ''
            display(Markdown(
                f'_line {line_no}{encl_note}_\n```c\n{ctx}\n```'
            ))